# READ THE BRONZE DATA CREATE DATA FRAME 

In [2]:
customers_raw = spark.read.parquet("abfss://352b8d60-3e18-4e21-93f5-6997945b481b@onelake.dfs.fabric.microsoft.com/42d8b084-acb7-4e20-b58c-97a29860b82a/Files/Bronze/customers.parquet")
orders_raw = spark.read.parquet("abfss://352b8d60-3e18-4e21-93f5-6997945b481b@onelake.dfs.fabric.microsoft.com/42d8b084-acb7-4e20-b58c-97a29860b82a/Files/Bronze/orders.parquet")
payments_raw = spark.read.parquet("abfss://352b8d60-3e18-4e21-93f5-6997945b481b@onelake.dfs.fabric.microsoft.com/42d8b084-acb7-4e20-b58c-97a29860b82a/Files/Bronze/payments.parquet")
support_raw = spark.read.parquet("abfss://352b8d60-3e18-4e21-93f5-6997945b481b@onelake.dfs.fabric.microsoft.com/42d8b084-acb7-4e20-b58c-97a29860b82a/Files/Bronze/support_tickets.parquet")
web_raw = spark.read.parquet("abfss://352b8d60-3e18-4e21-93f5-6997945b481b@onelake.dfs.fabric.microsoft.com/42d8b084-acb7-4e20-b58c-97a29860b82a/Files/Bronze/web_activities.parquet")

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 4, Finished, Available, Finished, False)

# read bronze data

In [3]:
display(customers_raw.limit(5))

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ae1d0e6c-66c8-4c20-b9b7-a5a55fbe7041)

In [4]:
display(orders_raw.limit(5))

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c3f6c9bc-e68a-43b2-81bb-3bc26f74edf4)

In [5]:
display(payments_raw.limit(5))

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c540393e-0749-42b0-b81d-faf1cf4dbaa7)

In [6]:
display(support_raw.limit(5))

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fda0bcb8-af81-43d8-ac98-d9acb6b7496a)

In [7]:
display(web_raw.limit(5))

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 30901c21-9f0d-4035-b710-21d70df5b432)

# Create delta bronze table

In [8]:
# Save as Bronze Delta Tables
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_raw.write.format("delta").mode("overwrite").saveAsTable("web")

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 10, Finished, Available, Finished, False)

# cleaned the data - silver 

## customer data -clean

In [9]:
display(customers_raw.limit(5))

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 384ddc62-c1d3-480b-8b6f-3af998484f8e)

In [10]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)
display(customers_clean.limit(6))

customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")



StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f24d1736-f6c3-4e00-89b2-7cd70fb92e3f)

## clean order table

In [11]:
%%sql 
select * from orders limit 5

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 13, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [12]:
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date", 
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
display(orders_clean.limit(5))
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e04106e6-7cb5-4c1c-a8e3-8394668439a3)

# Payments table - clean

In [15]:
%%sql 
select * from payments limit 5

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 17, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>

In [14]:
payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
display(payments_clean.limit(5))
payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")


StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3c505a5a-13e7-404e-81e4-35a436cf6cba)

# clean - support tables

In [16]:
%%sql
select * from support limit 5

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 18, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [17]:
support = spark.table("support")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))

support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 80abd500-2d42-4b85-9473-7003dff12365)

# - clean - web data 

In [18]:
%%sql 
select * from web limit 5

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 20, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [19]:
web = spark.table("web")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")


StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48c641f5-6d65-4aed-941c-14da49cd8ef0)

# GOLD TABLES - AGGRGATE TABLES

In [20]:
cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status").alias("order_status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)
display(customer360.limit(10))

customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, d47aee31-2f98-4e60-b002-0dffc15a96c4, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2f0d06fb-464a-4b65-ba9e-369d2ee82d29)